Web scratching all the files of the CSB Website.

In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time

# Base URL for the microdata catalog
catalog_url = 'https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata'

def get_category_links(catalog_url):
    try:
        response = requests.get(catalog_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching catalog page: {e}')
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    category_links = set()

    # Find the links to categories
    for link in soup.find_all('a', href=True):
        href = link['href']
        # Complete relative URLs
        full_url = urljoin(catalog_url, href)
        # Filter links that are categories
        if '/catalogus-microdata/' in full_url and full_url != catalog_url:
            category_links.add(full_url)

    print(f'Found {len(category_links)} category links.')
    return list(category_links)

# Get category links
category_links = get_category_links(catalog_url)


Found 19 category links.


In [17]:
def get_dataset_links_from_category(category_url):
    try:
        response = requests.get(category_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching category page {category_url}: {e}')
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    dataset_links = set()

    # Find links to dataset pages
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(category_url, href)
        # Filter links that are dataset pages
        if '/microdatabestanden/' in full_url:
            dataset_links.add(full_url)

    print(f'Found {len(dataset_links)} dataset links in category {category_url}.')
    return list(dataset_links)

# Collect all dataset links from all categories
all_dataset_links = set()
for category_url in category_links:
    dataset_links = get_dataset_links_from_category(category_url)
    all_dataset_links.update(dataset_links)
    # Optional delay to be polite
    time.sleep(1)

print(f'Total dataset links collected: {len(all_dataset_links)}')

# Save dataset links to a file (optional)
with open('dataset_links.txt', 'w', encoding='utf-8') as f:
    for link in all_dataset_links:
        f.write(f'{link}\n')


Found 14 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/landbouw.
Found 3 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/handel-en-horeca.
Found 50 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/bouwen-en-wonen.
Found 15 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/industrie-en-energie.
Found 46 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/bedrijven.
Found 2 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/internationale-handel.
Found 44 dataset links in

In [19]:
import re

def get_pdf_links(dataset_url):
    try:
        response = requests.get(dataset_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching dataset page {dataset_url}: {e}')
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    pdf_links = set()
    pdf_pattern = re.compile(r'\.pdf$', re.IGNORECASE)

    # Find all links that are PDFs
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(dataset_url, href)
        if pdf_pattern.search(full_url):
            pdf_links.add(full_url)
            continue
        if 'pdf' in link.get_text().lower():
            pdf_links.add(full_url)

    return list(pdf_links)


In [20]:
def download_pdf(pdf_url, save_dir='cbs_pdfs'):
    pdf_name = pdf_url.split('/')[-1]
    pdf_path = os.path.join(save_dir, pdf_name)
    if os.path.exists(pdf_path):
        print(f'Already downloaded: {pdf_name}')
        return

    try:
        response = requests.get(pdf_url)
        response.raise_for_status()
        with open(pdf_path, 'wb') as f:
            f.write(response.content)
        print(f'Downloaded: {pdf_name}')
    except requests.exceptions.RequestException as e:
        print(f'Error downloading {pdf_url}: {e}')


In [21]:
def main():
    # Step 1: Get category links
    category_links = get_category_links(catalog_url)

    # Step 2: Get dataset links from each category
    all_dataset_links = set()
    for category_url in category_links:
        dataset_links = get_dataset_links_from_category(category_url)
        all_dataset_links.update(dataset_links)
        # Optional delay
        time.sleep(1)

    print(f'Total dataset links collected: {len(all_dataset_links)}')

    # Save dataset links to a file
    with open('dataset_links.txt', 'w', encoding='utf-8') as f:
        for link in all_dataset_links:
            f.write(f'{link}\n')

    # Step 3: Extract and download PDFs from each dataset page
    os.makedirs('cbs_pdfs', exist_ok=True)
    total_pdfs = set()

    for dataset_url in all_dataset_links:
        print(f'\nProcessing dataset page: {dataset_url}')
        pdf_links = get_pdf_links(dataset_url)
        print(f'Found {len(pdf_links)} PDF(s) on this page.')
        total_pdfs.update(pdf_links)

        for pdf_url in pdf_links:
            download_pdf(pdf_url)

        # Optional delay
        time.sleep(1)

    print(f'\nTotal PDFs collected: {len(total_pdfs)}')

if __name__ == '__main__':
    main()


Found 19 category links.
Found 14 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/landbouw.
Found 3 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/handel-en-horeca.
Found 50 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/bouwen-en-wonen.
Found 15 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/industrie-en-energie.
Found 46 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/bedrijven.
Found 2 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/internationale-handel.


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Found 0 PDF(s) on this page.

Processing dataset page: https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/microdatabestanden/gbapersoontab-persoonskenmerken-van-personen-in-de-brp
Found 1 PDF(s) on this page.
Downloaded: gbapersoontab.pdf

Processing dataset page: https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/microdatabestanden/adoptiekinderen-geadopteerde-personen
Found 0 PDF(s) on this page.

Processing dataset page: https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/microdatabestanden/diplomavavotab-behaalde-vavo-diploma-s
Found 1 PDF(s) on this page.
Downloaded: diplomavavotab.pdf

Processing dataset page: https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/microdatabestanden/lmo2022-learning-mobility
Found 1 PDF(s) on this page.
Downloaded: lmo2022.pdf

Processing dataset page: https://www.cbs.nl/nl-nl/onze-diensten/maa

In [ ]:
#The whole thing:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time

# Base URL for the microdata catalog
catalog_url = 'https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata'

def get_category_links(catalog_url):
    try:
        response = requests.get(catalog_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching catalog page: {e}')
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    category_links = set()

    # Find the links to categories
    for link in soup.find_all('a', href=True):
        href = link['href']
        # Complete relative URLs
        full_url = urljoin(catalog_url, href)
        # Filter links that are categories
        if '/catalogus-microdata/' in full_url and full_url != catalog_url:
            category_links.add(full_url)

    print(f'Found {len(category_links)} category links.')
    return list(category_links)

# Get category links
category_links = get_category_links(catalog_url)

def get_dataset_links_from_category(category_url):
    try:
        response = requests.get(category_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching category page {category_url}: {e}')
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    dataset_links = set()

    # Find links to dataset pages
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(category_url, href)
        # Filter links that are dataset pages
        if '/microdatabestanden/' in full_url:
            dataset_links.add(full_url)

    print(f'Found {len(dataset_links)} dataset links in category {category_url}.')
    return list(dataset_links)

# Collect all dataset links from all categories
all_dataset_links = set()
for category_url in category_links:
    dataset_links = get_dataset_links_from_category(category_url)
    all_dataset_links.update(dataset_links)
    # Optional delay to be polite
    time.sleep(1)

print(f'Total dataset links collected: {len(all_dataset_links)}')

# Save dataset links to a file (optional)
with open('dataset_links.txt', 'w', encoding='utf-8') as f:
    for link in all_dataset_links:
        f.write(f'{link}\n')

import re

def get_pdf_links(dataset_url):
    try:
        response = requests.get(dataset_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching dataset page {dataset_url}: {e}')
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    pdf_links = set()
    pdf_pattern = re.compile(r'\.pdf$', re.IGNORECASE)

    # Find all links that are PDFs
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(dataset_url, href)
        if pdf_pattern.search(full_url):
            pdf_links.add(full_url)
            continue
        if 'pdf' in link.get_text().lower():
            pdf_links.add(full_url)

    return list(pdf_links)

def download_pdf(pdf_url, save_dir='cbs_pdfs'):
    pdf_name = pdf_url.split('/')[-1]
    pdf_path = os.path.join(save_dir, pdf_name)
    if os.path.exists(pdf_path):
        print(f'Already downloaded: {pdf_name}')
        return

    try:
        response = requests.get(pdf_url)
        response.raise_for_status()
        with open(pdf_path, 'wb') as f:
            f.write(response.content)
        print(f'Downloaded: {pdf_name}')
    except requests.exceptions.RequestException as e:
        print(f'Error downloading {pdf_url}: {e}')

def main():
    # Step 1: Get category links
    category_links = get_category_links(catalog_url)

    # Step 2: Get dataset links from each category
    all_dataset_links = set()
    for category_url in category_links:
        dataset_links = get_dataset_links_from_category(category_url)
        all_dataset_links.update(dataset_links)
        # Optional delay
        time.sleep(1)

    print(f'Total dataset links collected: {len(all_dataset_links)}')

    # Save dataset links to a file
    with open('dataset_links.txt', 'w', encoding='utf-8') as f:
        for link in all_dataset_links:
            f.write(f'{link}\n')

    # Step 3: Extract and download PDFs from each dataset page
    os.makedirs('cbs_pdfs', exist_ok=True)
    total_pdfs = set()

    for dataset_url in all_dataset_links:
        print(f'\nProcessing dataset page: {dataset_url}')
        pdf_links = get_pdf_links(dataset_url)
        print(f'Found {len(pdf_links)} PDF(s) on this page.')
        total_pdfs.update(pdf_links)

        for pdf_url in pdf_links:
            download_pdf(pdf_url)

        # Optional delay
        time.sleep(1)

    print(f'\nTotal PDFs collected: {len(total_pdfs)}')

if __name__ == '__main__':
    main()



SyntaxError: invalid syntax (3207174852.py, line 1)

We actually used this one

In [23]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time
import re

# Base URL for the microdata catalog
catalog_url = 'https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata'

def sanitize_filename(name):
    # Remove invalid characters for filenames and folder names
    return re.sub(r'[\\/*?:"<>|]', "_", name)

def get_category_links(catalog_url):
    try:
        response = requests.get(catalog_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching catalog page: {e}')
        return {}

    soup = BeautifulSoup(response.content, 'html.parser')
    category_links = {}

    # Find the links to categories
    for link in soup.find_all('a', href=True):
        href = link['href']
        category_name = link.get_text().strip()
        # Complete relative URLs
        full_url = urljoin(catalog_url, href)
        # Filter links that are categories
        if '/catalogus-microdata/' in full_url and full_url != catalog_url:
            sanitized_category_name = sanitize_filename(category_name)
            category_links[sanitized_category_name] = full_url

    print(f'Found {len(category_links)} category links.')
    return category_links

def get_dataset_links_from_category(category_url):
    try:
        response = requests.get(category_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching category page {category_url}: {e}')
        return {}

    soup = BeautifulSoup(response.content, 'html.parser')
    dataset_links = {}

    # Find links to dataset pages
    for link in soup.find_all('a', href=True):
        href = link['href']
        dataset_name = link.get_text().strip()
        full_url = urljoin(category_url, href)
        # Filter links that are dataset pages
        if '/microdatabestanden/' in full_url:
            sanitized_dataset_name = sanitize_filename(dataset_name)
            dataset_links[sanitized_dataset_name] = full_url

    print(f'Found {len(dataset_links)} dataset links in category {category_url}.')
    return dataset_links

def get_pdf_links(dataset_url):
    try:
        response = requests.get(dataset_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching dataset page {dataset_url}: {e}')
        return {}

    soup = BeautifulSoup(response.content, 'html.parser')
    pdf_links = {}
    pdf_pattern = re.compile(r'\.pdf$', re.IGNORECASE)

    # Find all links that are PDFs
    for link in soup.find_all('a', href=True):
        href = link['href']
        pdf_name = link.get_text().strip() or href.split('/')[-1]
        full_url = urljoin(dataset_url, href)
        if pdf_pattern.search(full_url) or 'pdf' in link.get_text().lower():
            sanitized_pdf_name = sanitize_filename(pdf_name)
            pdf_links[sanitized_pdf_name] = full_url

    return pdf_links

def download_pdf(pdf_url, pdf_name, save_dir):
    pdf_path = os.path.join(save_dir, pdf_name)
    if os.path.exists(pdf_path):
        print(f'Already downloaded: {pdf_path}')
        return

    try:
        response = requests.get(pdf_url)
        response.raise_for_status()
        with open(pdf_path, 'wb') as f:
            f.write(response.content)
        print(f'Downloaded: {pdf_path}')
    except requests.exceptions.RequestException as e:
        print(f'Error downloading {pdf_url}: {e}')

def main():
    # Step 1: Get category links
    category_links = get_category_links(catalog_url)

    # Base directory for saving PDFs
    base_dir = 'cbs_pdfs'
    os.makedirs(base_dir, exist_ok=True)

    # Step 2: Get dataset links from each category
    total_pdfs = 0
    for category_name, category_url in category_links.items():
        print(f'\nProcessing category: {category_name}')
        category_dir = os.path.join(base_dir, category_name)
        os.makedirs(category_dir, exist_ok=True)

        dataset_links = get_dataset_links_from_category(category_url)
        time.sleep(1)  # Optional delay

        # Step 3: Extract and download PDFs from each dataset page
        for dataset_name, dataset_url in dataset_links.items():
            print(f'Processing dataset: {dataset_name}')
            dataset_dir = os.path.join(category_dir, dataset_name)
            os.makedirs(dataset_dir, exist_ok=True)

            pdf_links = get_pdf_links(dataset_url)
            time.sleep(1)  # Optional delay

            if pdf_links:
                for pdf_name, pdf_url in pdf_links.items():
                    download_pdf(pdf_url, pdf_name + '.pdf', dataset_dir)
                    total_pdfs += 1
            else:
                print(f'No PDFs found for dataset {dataset_name}.')

    print(f'\nTotal PDFs downloaded: {total_pdfs}')

if __name__ == '__main__':
    main()


Found 19 category links.

Processing category: Arbeid en sociale zekerheid
Found 147 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/arbeid-en-sociale-zekerheid.
Processing dataset: Aanvulvergoedpersoonmndbedragbus_ Mndbedrgn aavulvergd
Downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Aanvulvergoedpersoonmndbedragbus_ Mndbedrgn aavulvergd/Aanvulvergoedpersoonmndbedragbus.pdf
Processing dataset: Akwuitkering1atab_ Kenmerken van AKW-uitkeringen
Downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Akwuitkering1atab_ Kenmerken van AKW-uitkeringen/Akwuitkering1atab.pdf
Processing dataset: Akwuitkering1btab_ AKW-uitkeringen van kinderen
Downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Akwuitkering1btab_ AKW-uitkeringen van kinderen/Akwuitkering1btab.pdf
Processing dataset: Anw_ Maandbedragen ANW-uitkering
No PDFs found for dataset Anw_ Maandbedragen ANW-uitkering.
Processing dataset: Anwjaarbedragtab_ Jaarbe

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


No PDFs found for dataset Gebruikershandleiding Onderwijsinformatie.
Processing dataset: Gpe_  Gepromoveerden Enquête
Downloaded: cbs_pdfs/Onderwijs/Gpe_  Gepromoveerden Enquête/Gpe.pdf
Processing dataset: Hdiplomahbotab_ Kenmerken hoofddiploma's HBO
Downloaded: cbs_pdfs/Onderwijs/Hdiplomahbotab_ Kenmerken hoofddiploma's HBO/Hdiplomahbotab.pdf
Processing dataset: Hdiplomaregtab_ Hoogst behaalde diploma
Downloaded: cbs_pdfs/Onderwijs/Hdiplomaregtab_ Hoogst behaalde diploma/Hdiplomaregtab.pdf
Processing dataset: Hdiplomawotab_ Kenmerken hoofddiploma's WO
Downloaded: cbs_pdfs/Onderwijs/Hdiplomawotab_ Kenmerken hoofddiploma's WO/hdiplomawotab.pdf
Processing dataset: HO_ Inschrijvingen en diploma's hoger onderwijs
No PDFs found for dataset HO_ Inschrijvingen en diploma's hoger onderwijs.
Processing dataset: Hoogsteopltab_ Hoogst behaald_gevolgd opleidingsniveau
Downloaded: cbs_pdfs/Onderwijs/Hoogsteopltab_ Hoogst behaald_gevolgd opleidingsniveau/Hoogsteopltab.pdf
Processing dataset: Inschrw

We used this as well - this creates with folders per section

In [46]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time
import re

# Base URL for the microdata catalog
catalog_url = 'https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata'

def sanitize_filename(name):
    # Remove invalid characters for filenames and folder names
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    name = name.strip()
    return name

def get_category_links(catalog_url):
    try:
        response = requests.get(catalog_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching catalog page: {e}')
        return {}

    soup = BeautifulSoup(response.content, 'html.parser')
    category_links = {}

    # Find the links to categories
    for link in soup.find_all('a', href=True):
        href = link['href']
        category_name = link.get_text().strip()
        # Complete relative URLs
        full_url = urljoin(catalog_url, href)
        # Filter links that are categories
        if '/catalogus-microdata/' in full_url and full_url != catalog_url:
            sanitized_category_name = sanitize_filename(category_name)
            category_links[sanitized_category_name] = full_url

    print(f'Found {len(category_links)} category links.')
    return category_links

def get_dataset_links_from_category(category_url):
    try:
        response = requests.get(category_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching category page {category_url}: {e}')
        return {}

    soup = BeautifulSoup(response.content, 'html.parser')
    dataset_links = {}

    # Find links to dataset pages
    for link in soup.find_all('a', href=True):
        href = link['href']
        dataset_name = link.get_text().strip()
        full_url = urljoin(category_url, href)
        # Filter links that are dataset pages
        if '/microdatabestanden/' in full_url:
            sanitized_dataset_name = sanitize_filename(dataset_name)
            dataset_links[sanitized_dataset_name] = full_url

    print(f'Found {len(dataset_links)} dataset links in category {category_url}.')
    return dataset_links

def get_pdf_links(dataset_url):
    try:
        response = requests.get(dataset_url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Error fetching dataset page {dataset_url}: {e}')
        return {}

    soup = BeautifulSoup(response.content, 'html.parser')
    pdf_links = {}
    pdf_pattern = re.compile(r'\.pdf$', re.IGNORECASE)

    # Find all links that are PDFs
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(dataset_url, href)
        pdf_name = link.get_text().strip()
        if not pdf_name:
            # Use the last part of the URL if link text is empty
            pdf_name = href.split('/')[-1]

        sanitized_pdf_name = sanitize_filename(pdf_name)
        # Ensure pdf_name does not contain path separators
        sanitized_pdf_name = sanitized_pdf_name.replace('/', '_').replace('\\', '_')

        if pdf_pattern.search(full_url) or 'pdf' in link.get_text().lower():
            pdf_links[sanitized_pdf_name] = full_url

    return pdf_links

def download_pdf(pdf_url, pdf_name, save_dir):
    pdf_name_with_ext = pdf_name
    if not pdf_name_with_ext.lower().endswith('.pdf'):
        pdf_name_with_ext += '.pdf'

    pdf_path = os.path.join(save_dir, pdf_name_with_ext)
    if os.path.exists(pdf_path):
        print(f'Already downloaded: {pdf_path}')
        return

    try:
        response = requests.get(pdf_url)
        response.raise_for_status()
        with open(pdf_path, 'wb') as f:
            f.write(response.content)
        print(f'Downloaded: {pdf_path}')
    except requests.exceptions.RequestException as e:
        print(f'Error downloading {pdf_url}: {e}')

def main():
    # Step 1: Get category links
    category_links = get_category_links(catalog_url)

    # Base directory for saving PDFs
    base_dir = 'cbs_pdfs'
    os.makedirs(base_dir, exist_ok=True)

    # Step 2: Get dataset links from each category
    total_pdfs = 0
    for category_name, category_url in category_links.items():
        print(f'\nProcessing category: {category_name}')
        category_dir = os.path.join(base_dir, category_name)
        os.makedirs(category_dir, exist_ok=True)

        dataset_links = get_dataset_links_from_category(category_url)
        time.sleep(1)  # Optional delay

        # Step 3: Extract and download PDFs from each dataset page
        for dataset_name, dataset_url in dataset_links.items():
            print(f'Processing dataset: {dataset_name}')
            dataset_dir = category_dir  # Save PDFs directly in the category folder
            # If you prefer to have dataset folders, uncomment the following lines:
            # dataset_dir = os.path.join(category_dir, dataset_name)
            # os.makedirs(dataset_dir, exist_ok=True)

            pdf_links = get_pdf_links(dataset_url)
            time.sleep(1)  # Optional delay

            if pdf_links:
                for pdf_name, pdf_url in pdf_links.items():
                    download_pdf(pdf_url, pdf_name, dataset_dir)
                    total_pdfs += 1
            else:
                print(f'No PDFs found for dataset {dataset_name}.')

    print(f'\nTotal PDFs downloaded: {total_pdfs}')

if __name__ == '__main__':
    main()


Found 19 category links.

Processing category: Arbeid en sociale zekerheid
Found 147 dataset links in category https://www.cbs.nl/nl-nl/onze-diensten/maatwerk-en-microdata/microdata-zelf-onderzoek-doen/catalogus-microdata/arbeid-en-sociale-zekerheid.
Processing dataset: Aanvulvergoedpersoonmndbedragbus_ Mndbedrgn aavulvergd
Already downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Aanvulvergoedpersoonmndbedragbus.pdf
Processing dataset: Akwuitkering1atab_ Kenmerken van AKW-uitkeringen
Already downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Akwuitkering1atab.pdf
Processing dataset: Akwuitkering1btab_ AKW-uitkeringen van kinderen
Already downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Akwuitkering1btab.pdf
Processing dataset: Anw_ Maandbedragen ANW-uitkering
No PDFs found for dataset Anw_ Maandbedragen ANW-uitkering.
Processing dataset: Anwjaarbedragtab_ Jaarbedragen ANW-uitkering
Already downloaded: cbs_pdfs/Arbeid en sociale zekerheid/Anwjaarbedragtab.pdf
Processing dataset: Anwmndbed

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


No PDFs found for dataset Gebruikershandleiding Onderwijsinformatie.
Processing dataset: Gpe_  Gepromoveerden Enquête
Downloaded: cbs_pdfs/Onderwijs/Gpe.pdf
Processing dataset: Hdiplomahbotab_ Kenmerken hoofddiploma's HBO
Downloaded: cbs_pdfs/Onderwijs/Hdiplomahbotab.pdf
Processing dataset: Hdiplomaregtab_ Hoogst behaalde diploma
Downloaded: cbs_pdfs/Onderwijs/Hdiplomaregtab.pdf
Processing dataset: Hdiplomawotab_ Kenmerken hoofddiploma's WO
Downloaded: cbs_pdfs/Onderwijs/hdiplomawotab.pdf
Processing dataset: HO_ Inschrijvingen en diploma's hoger onderwijs
No PDFs found for dataset HO_ Inschrijvingen en diploma's hoger onderwijs.
Processing dataset: Hoogsteopltab_ Hoogst behaald_gevolgd opleidingsniveau
Downloaded: cbs_pdfs/Onderwijs/Hoogsteopltab.pdf
Processing dataset: Inschrwectab_  Inschrijvingen van leerlingen in het speciaal onderwijs
Downloaded: cbs_pdfs/Onderwijs/Inschrwectab.pdf
Processing dataset: Inschrwpotab_ Inschrijvingen in het basisonderwijs
Downloaded: cbs_pdfs/Onderwij

Processing the 500 pdfs!!!

In [35]:
import pdfplumber
import os

pdf_folder = 'cbs_pdfs_all'
output_folder = 'extracted_texts'
os.makedirs(output_folder, exist_ok=True)

for pdf_file in os.listdir(pdf_folder):
    if pdf_file.endswith('.pdf'):
        pdf_path = os.path.join(pdf_folder, pdf_file)
        txt_file = pdf_file.replace('.pdf', '.txt')
        txt_path = os.path.join(output_folder, txt_file)
        try:
            with pdfplumber.open(pdf_path) as pdf:
                full_text = ''
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        full_text += text + '\n'
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(full_text)
            print(f'Extracted text from {pdf_file}')
        except Exception as e:
            print(f'Error processing {pdf_file}: {e}')


Extracted text from dementiepopulatievektistab.pdf
Extracted text from gpe.pdf
Extracted text from schtab.pdf
Extracted text from lsb2022.pdf
Extracted text from zea2023.pdf
Extracted text from eu-silc-2015.pdf
Extracted text from ovarbeidbus.pdf
Extracted text from doodoorztab.pdf
Extracted text from abrvanaf2023.pdf
Extracted text from nea2014-2022.pdf
Extracted text from pensnabestpersoonmndbedragbus.pdf
Extracted text from inhatab.pdf
Extracted text from hjlktab.pdf
Extracted text from vrlmigmotbus.pdf
Extracted text from deelcertificatenvavotab.pdf
Extracted text from opbouw-en-instructie-totaalbestand-gezondheidsmonitor-volwassenen-en-ouderen-2022_v3.pdf
Extracted text from deeltijdfactorbaanbus.pdf
Extracted text from uitstroomomtab.pdf
Extracted text from rktontvangerstab.pdf
Extracted text from finmon.pdf
Extracted text from indicawbztab.pdf
Extracted text from bedrijfsopleidingen2015.pdf
Extracted text from onderwijsinschrtab.pdf
Extracted text from cco2017_2018.pdf
Extracted

In [36]:
import re

variable_pattern = re.compile(r'^\d+\s+(.+)', re.MULTILINE)


In [37]:
import re

def parse_variables_from_text(text):
    variables = []
    lines = text.split('\n')
    in_variables_section = False
    current_variable = {}

    for line in lines:
        line = line.strip()
        if 'Toelichting op de variabelen' in line:
            in_variables_section = True
            continue
        if 'Versiegeschiedenis' in line:
            in_variables_section = False
            continue
        if in_variables_section:
            # Match variable name and description
            if re.match(r'^\d+\s+\w+', line):
                if current_variable:
                    variables.append(current_variable)
                parts = line.split()
                var_number = parts[0]
                var_name = ' '.join(parts[1:])
                current_variable = {'variable_name': var_name, 'definition': '', 'explanation': ''}
            else:
                # Accumulate definition and explanation
                if 'Definitie' in line:
                    current_variable['definition'] = ''
                elif 'Toelichting bij de definitie' in line or 'Toelichting bij het gebruik' in line:
                    current_variable['explanation'] = ''
                else:
                    if 'definition' in current_variable and current_variable['definition'] == '':
                        current_variable['definition'] += line + ' '
                    elif 'explanation' in current_variable and current_variable['explanation'] == '':
                        current_variable['explanation'] += line + ' '
    # Add the last variable
    if current_variable:
        variables.append(current_variable)
    return variables


In [38]:
import json

variables_list = []

for txt_file in os.listdir(output_folder):
    if txt_file.endswith('.txt'):
        txt_path = os.path.join(output_folder, txt_file)
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()
        variables = parse_variables_from_text(text)
        for var in variables:
            var['source_pdf'] = txt_file.replace('.txt', '.pdf')
            variables_list.append(var)

# Save to JSON or CSV
with open('variables.json', 'w', encoding='utf-8') as f:
    json.dump(variables_list, f, ensure_ascii=False, indent=4)


In [34]:
import json

variables_list = []

for txt_file in os.listdir(output_folder):
    if txt_file.endswith('.txt'):
        txt_path = os.path.join(output_folder, txt_file)
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()
        variables = parse_variables_from_text(text)
        for var in variables:
            var['source_pdf'] = txt_file.replace('.txt', '.pdf')
            variables_list.append(var)

# Save to JSON or CSV
with open('variables.json', 'w', encoding='utf-8') as f:
    json.dump(variables_list, f, ensure_ascii=False, indent=4)


In [52]:
import os
import glob

# Base directory
pdf_base_dir = 'cbs_pdfs'
#cbs_pdfs_with_folders'

# Find all PDF files recursively
pdf_files = glob.glob(os.path.join(pdf_base_dir, '**', '*.pdf'), recursive=True)

In [53]:
pdf_files

['cbs_pdfs/Handel en horeca/Ps detailhandel.pdf',
 'cbs_pdfs/Handel en horeca/Ps autohandel.pdf',
 'cbs_pdfs/Handel en horeca/Ps groothandel.pdf',
 'cbs_pdfs/Financiële en zakelijke diensten/Ps commerciele dienstverlening.pdf',
 'cbs_pdfs/Financiële en zakelijke diensten/Procedure lisspanel CBS.pdf',
 'cbs_pdfs/Landbouw/Lsb2022.pdf',
 'cbs_pdfs/Landbouw/Lsb2021.pdf',
 'cbs_pdfs/Landbouw/lsb2020.pdf',
 'cbs_pdfs/Landbouw/lsb2018.pdf',
 'cbs_pdfs/Landbouw/lsb2019.pdf',
 'cbs_pdfs/Landbouw/Oogstrakker microdata.pdf',
 'cbs_pdfs/Landbouw/Ps dienstverleninglandbouw.pdf',
 'cbs_pdfs/Landbouw/Gewaspercelen 2017-2019.pdf',
 'cbs_pdfs/Landbouw/Lbt_fd.pdf',
 'cbs_pdfs/Landbouw/Landbouwtellingen.pdf',
 'cbs_pdfs/Landbouw/lsb2017.pdf',
 'cbs_pdfs/Landbouw/lsb2016.pdf',
 'cbs_pdfs/Landbouw/lsb2015.pdf',
 'cbs_pdfs/Landbouw/Gewaspercelen vanaf 2020.pdf',
 'cbs_pdfs/Veiligheid en recht/Hjlktab.pdf',
 'cbs_pdfs/Veiligheid en recht/Uitstroomomtab.pdf',
 'cbs_pdfs/Veiligheid en recht/Gedetineerdentab.

In [47]:
for pdf_file in pdf_files:
    # Normalize the path
    pdf_path = os.path.normpath(pdf_file)
    # Split the path into parts
    path_parts = pdf_path.split(os.sep)
    # Find the index of the base directory
    try:
        base_index = path_parts.index(os.path.basename(pdf_base_dir))
    except ValueError:
        # If not found, assume the base directory is the current working directory
        base_index = 0
    # Extract category and dataset names
    if len(path_parts) > base_index + 2:
        category_name = path_parts[base_index + 1]
        dataset_name = path_parts[base_index + 2]
    elif len(path_parts) > base_index + 1:
        category_name = path_parts[base_index + 1]
        dataset_name = 'Unknown Dataset'
    else:
        category_name = 'Unknown Category'
        dataset_name = 'Unknown Dataset'
    print(f"PDF: {pdf_file}, Category: {category_name}, Dataset: {dataset_name}")


PDF: cbs_pdfs_with_folders/Handel en horeca/Ps Detailhandel_ productiestatistiek detailhandel/Ps detailhandel.pdf, Category: Handel en horeca, Dataset: Ps Detailhandel_ productiestatistiek detailhandel
PDF: cbs_pdfs_with_folders/Handel en horeca/Ps Groothandel_  productiestatistiek groothandel/Ps groothandel.pdf, Category: Handel en horeca, Dataset: Ps Groothandel_  productiestatistiek groothandel
PDF: cbs_pdfs_with_folders/Handel en horeca/Ps Autohandel_ productiestatistiek Autohandel/Ps autohandel.pdf, Category: Handel en horeca, Dataset: Ps Autohandel_ productiestatistiek Autohandel
PDF: cbs_pdfs_with_folders/Financiële en zakelijke diensten/LISSPANEL_ procedure Lisspanel met CBS microdata/Procedure lisspanel CBS.pdf, Category: Financiële en zakelijke diensten, Dataset: LISSPANEL_ procedure Lisspanel met CBS microdata
PDF: cbs_pdfs_with_folders/Financiële en zakelijke diensten/Ps CD_ Productiestatistiek Commerciële dienstverlening/Ps commerciele dienstverlening.pdf, Category: Fi

In [77]:
pdf_to_dataset = {}

for pdf_file in pdf_files:
    pdf_path = os.path.normpath(pdf_file)
    path_parts = pdf_path.split(os.sep)
    # Find the index of the base directory
    try:
        base_index = path_parts.index(os.path.basename(pdf_base_dir))
    except ValueError:
        base_index = 0
    # Extract dataset name
    if len(path_parts) > base_index + 1:
        dataset_name = path_parts[base_index + 1]
    else:
        dataset_name = 'Unknown Dataset'
    # Get the PDF file name
    pdf_filename = os.path.basename(pdf_file)
    pdf_to_dataset[pdf_filename] = dataset_name


In [78]:
for var in variables_list:
    pdf_filename = var['source_pdf']
    # Get the dataset name from the mapping
    dataset_name = pdf_to_dataset.get(pdf_filename, 'Unknown Dataset')
    var['dataset_name'] = dataset_name


In [79]:
for var in variables_list[:10]:  # Check the first 10 entries
    print(f"Variable: {var['variable_name']}, Dataset: {var['dataset_name']}")


Variable: BE_Id F11, Dataset: Unknown Dataset
Variable: STATJAAR F4, Dataset: Unknown Dataset
Variable: sbi A6, Dataset: Unknown Dataset
Variable: kerncelcode A6, Dataset: Unknown Dataset
Variable: RechtsvormID F3, Dataset: Unknown Dataset
Variable: gk_SBS F6, Dataset: Unknown Dataset
Variable: EindGewicht F8.2, Dataset: Unknown Dataset
Variable: CorrectieGewicht F8.2, Dataset: Unknown Dataset
Variable: Correctiefactor_DRT F8.2, Dataset: Unknown Dataset
Variable: ImputatieGebruiken A5, Dataset: Unknown Dataset


In [82]:
variables_list

[{'variable_name': 'BE_Id F11',
  'definition': 'Bedrijfsindentificatie ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Unknown Parent Folder',
  'grandparent_folder': 'Unknown Grandparent Folder'},
 {'variable_name': 'STATJAAR F4',
  'definition': 'Verslagjaar ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Unknown Parent Folder',
  'grandparent_folder': 'Unknown Grandparent Folder'},
 {'variable_name': 'sbi A6',
  'definition': 'Standaard Bedrijfs Indeling 2008 code voor economische activiteit ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Unknown Parent Folder',
  'grandparent_folder': 'Unknown Grandparent Folder'},
 {'variable_name': 'kerncelcode A6',
  'definition': 'Code van de Kerncel ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Data

In [61]:
import os
import glob

pdf_base_dir = 'cbs_pdfs_with_folders'

pdf_files = glob.glob(os.path.join(pdf_base_dir, '**', '*.pdf'), recursive=True)

for pdf_file in pdf_files[:10]:  # Print first 10 PDF paths
    print(pdf_file)


cbs_pdfs_with_folders/Handel en horeca/Ps Detailhandel_ productiestatistiek detailhandel/Ps detailhandel.pdf
cbs_pdfs_with_folders/Handel en horeca/Ps Groothandel_  productiestatistiek groothandel/Ps groothandel.pdf
cbs_pdfs_with_folders/Handel en horeca/Ps Autohandel_ productiestatistiek Autohandel/Ps autohandel.pdf
cbs_pdfs_with_folders/Financiële en zakelijke diensten/LISSPANEL_ procedure Lisspanel met CBS microdata/Procedure lisspanel CBS.pdf
cbs_pdfs_with_folders/Financiële en zakelijke diensten/Ps CD_ Productiestatistiek Commerciële dienstverlening/Ps commerciele dienstverlening.pdf
cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2020/lsb2020.pdf
cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2018/lsb2018.pdf
cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2016/lsb2016.pdf
cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2017/lsb2017.pdf
cbs_pdfs_with_folders/Landbou

In [81]:
import os
import glob

pdf_base_dir = 'cbs_pdfs_with_folders'

# Step 1: Find all PDF files recursively
pdf_files = glob.glob(os.path.join(pdf_base_dir, '**', '*.pdf'), recursive=True)

# Step 2: Build the mapping
def normalize_filename(filename):
    filename = os.path.basename(filename)
    # Remove known prefixes
    prefixes = ['Documentatie_', 'Procedure ', 'Ps ', 'lsb']
    for prefix in prefixes:
        if filename.lower().startswith(prefix.lower()):
            filename = filename[len(prefix):]
            break
    # Convert to lowercase for case-insensitive matching
    filename = filename.lower()
    return filename

pdf_to_parents = {}

for pdf_file in pdf_files:
    # Normalize the path
    pdf_path = os.path.normpath(pdf_file)
    # Split the path into parts
    path_parts = pdf_path.split(os.sep)
    # Get the normalized PDF file name
    pdf_filename = normalize_filename(pdf_file)
    # The immediate parent folder (Dataset)
    if len(path_parts) >= 2:
        parent_folder = path_parts[-2]
    else:
        parent_folder = ''
    # The parent of the parent folder (Category)
    if len(path_parts) >= 3:
        grandparent_folder = path_parts[-3]
    else:
        grandparent_folder = ''
    # Store the information
    pdf_to_parents[pdf_filename] = {
        'parent_folder': parent_folder,
        'grandparent_folder': grandparent_folder
    }

# Step 3: Update variables_list
for var in variables_list:
    pdf_filename = normalize_filename(var['source_pdf'])
    parents = pdf_to_parents.get(pdf_filename)
    if parents:
        var['parent_folder'] = parents['parent_folder']
        var['grandparent_folder'] = parents['grandparent_folder']
    else:
        var['parent_folder'] = 'Unknown Parent Folder'
        var['grandparent_folder'] = 'Unknown Grandparent Folder'


In [83]:
variables_list

[{'variable_name': 'BE_Id F11',
  'definition': 'Bedrijfsindentificatie ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Unknown Parent Folder',
  'grandparent_folder': 'Unknown Grandparent Folder'},
 {'variable_name': 'STATJAAR F4',
  'definition': 'Verslagjaar ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Unknown Parent Folder',
  'grandparent_folder': 'Unknown Grandparent Folder'},
 {'variable_name': 'sbi A6',
  'definition': 'Standaard Bedrijfs Indeling 2008 code voor economische activiteit ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Unknown Parent Folder',
  'grandparent_folder': 'Unknown Grandparent Folder'},
 {'variable_name': 'kerncelcode A6',
  'definition': 'Code van de Kerncel ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Data

In [85]:
print("Keys in pdf_to_parents mapping:")
for key in pdf_to_parents.keys():
    print(key)


Keys in pdf_to_parents mapping:
detailhandel.pdf
groothandel.pdf
autohandel.pdf
lisspanel cbs.pdf
commerciele dienstverlening.pdf
2020.pdf
2018.pdf
2016.pdf
2017.pdf
2019.pdf
2021.pdf
lbt_fd.pdf
gewaspercelen 2017-2019.pdf
gewaspercelen vanaf 2020.pdf
2015.pdf
2022.pdf
oogstrakker microdata.pdf
landbouwtellingen.pdf
dienstverleninglandbouw.pdf
vm 2023.pdf
sfhulp.pdf
halttab.pdf
vm 2012.pdf
jgdbeschermbus.pdf
wsnptab.pdf
vm 2019.pdf
verdtab.pdf
phsg.pdf
phsg 2022.pdf
vm 2021.pdf
jgdreclasbus.pdf
wsnppersbus.pdf
vm 2013.pdf
incidentenbrandenenhulpverlening.pdf
sian.pdf
jgdhulpbus.pdf
instroomomtab.pdf
slotab.pdf
grbstab.pdf
standgedetineerdentab.pdf
mzj2023.pdf
hjlktab.pdf
vm 2016.pdf
mzj2020.pdf
gedetineerdentab.pdf
uitstroomrechtertab.pdf
vm 2015.pdf
dvc2018.pdf
sfbra.pdf
ovec.pdf
vm 2014.pdf
uitstroomomtab.pdf
vm 2017.pdf
shntab.pdf
bestaande koopwoningen.pdf
buitengewicht.pdf
huurenquete.pdf
woz.pdf
kenhuurcn.pdf
cpi.pdf
cpicn.pdf
abr.pdf
inward fats.pdf
onderwijs.pdf
zorgpraktijken.

In [86]:
print("Normalized pdf_filenames from variables_list:")
for var in variables_list[:10]:  # Adjust the range as needed
    pdf_filename = normalize_filename(var['source_pdf'])
    print(pdf_filename)


Normalized pdf_filenames from variables_list:
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf
ps-industrie.pdf


In [87]:
pdf_files

['cbs_pdfs_with_folders/Handel en horeca/Ps Detailhandel_ productiestatistiek detailhandel/Ps detailhandel.pdf',
 'cbs_pdfs_with_folders/Handel en horeca/Ps Groothandel_  productiestatistiek groothandel/Ps groothandel.pdf',
 'cbs_pdfs_with_folders/Handel en horeca/Ps Autohandel_ productiestatistiek Autohandel/Ps autohandel.pdf',
 'cbs_pdfs_with_folders/Financiële en zakelijke diensten/LISSPANEL_ procedure Lisspanel met CBS microdata/Procedure lisspanel CBS.pdf',
 'cbs_pdfs_with_folders/Financiële en zakelijke diensten/Ps CD_ Productiestatistiek Commerciële dienstverlening/Ps commerciele dienstverlening.pdf',
 'cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2020/lsb2020.pdf',
 'cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2018/lsb2018.pdf',
 'cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2016/lsb2016.pdf',
 'cbs_pdfs_with_folders/Landbouw/Lsb_ Structuurgegevens van landbouwbedrijven 2017/lsb2017

In [93]:
pip install ast

Defaulting to user installation because normal site-packages is not writeable
    ERROR: Command errored out with exit status 1:
     command: /Library/Developer/CommandLineTools/usr/bin/python3 -c 'import io, os, sys, setuptools, tokenize; sys.argv[0] = '"'"'/private/var/folders/32/8m2fx_7d0j73c3jp8nbjv0b00000gn/T/pip-install-cg8dlzh3/ast_ef3138fdadd948a3be3f956fb213ee12/setup.py'"'"'; __file__='"'"'/private/var/folders/32/8m2fx_7d0j73c3jp8nbjv0b00000gn/T/pip-install-cg8dlzh3/ast_ef3138fdadd948a3be3f956fb213ee12/setup.py'"'"';f = getattr(tokenize, '"'"'open'"'"', open)(__file__) if os.path.exists(__file__) else io.StringIO('"'"'from setuptools import setup; setup()'"'"');code = f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base /private/var/folders/32/8m2fx_7d0j73c3jp8nbjv0b00000gn/T/pip-pip-egg-info-7__i1_l9
         cwd: /private/var/folders/32/8m2fx_7d0j73c3jp8nbjv0b00000gn/T/pip-install-cg8dlzh3/ast_ef3138fda

In [ ]:
import ast 
# I want to use ist to separet each string in the list of pdf_files by / and then get the element -2 and -3, AND PRINT IT
pdf_files[0].split('/')
['cbs_pdfs_with_folders',
 'Handel en horeca',
 'Ps Detailhandel_ productiestatistiek detailhandel',
 'Ps detailhandel.pdf']

['cbs_pdfs_with_folders',
 'Handel en horeca',
 'Ps Detailhandel_ productiestatistiek detailhandel',
 'Ps detailhandel.pdf']

In [95]:
import os
import glob

# Base directory containing the PDFs
pdf_base_dir = 'cbs_pdfs_with_folders'

# Get the list of all PDF files
pdf_files = glob.glob(os.path.join(pdf_base_dir, '**', '*.pdf'), recursive=True)

# Create a mapping from normalized PDF filenames to their parent and grandparent folders
pdf_to_parents = {}

for pdf_file in pdf_files:
    # Normalize the path to use the correct OS-specific separators
    pdf_path = os.path.normpath(pdf_file)
    # Split the path into components
    path_parts = pdf_path.split(os.sep)
    # Get the PDF filename
    pdf_filename = os.path.basename(pdf_path)
    # Get the parent (-2) and grandparent (-3) folder names
    if len(path_parts) >= 3:
        parent_folder = path_parts[-2]
        grandparent_folder = path_parts[-3]
    else:
        parent_folder = 'Unknown Parent Folder'
        grandparent_folder = 'Unknown Grandparent Folder'
    # Add to the mapping
    pdf_to_parents[pdf_filename] = {
        'parent_folder': parent_folder,
        'grandparent_folder': grandparent_folder
    }

# Now, update your variables_list with the parent folders
for var in variables_list:
    pdf_filename = var['source_pdf']
    # Since the filenames may differ in case or have prefixes, normalize them if necessary
    # For simplicity, we'll assume they match exactly
    parents = pdf_to_parents.get(pdf_filename)
    if parents:
        var['parent_folder'] = parents['parent_folder']
        var['grandparent_folder'] = parents['grandparent_folder']
    else:
        var['parent_folder'] = 'Unknown Parent Folder'
        var['grandparent_folder'] = 'Unknown Grandparent Folder'


In [96]:
def normalize_filename(filename):
    filename = os.path.basename(filename)
    # Convert to lowercase for case-insensitive matching
    filename = filename.lower()
    # Remove known prefixes or suffixes if necessary
    prefixes = ['Documentatie_', 'Procedure ', 'Ps ', 'ps-', 'ps_', 'lsb']
    for prefix in prefixes:
        if filename.startswith(prefix.lower()):
            filename = filename[len(prefix):]
            break
    return filename

# Normalize keys in pdf_to_parents
pdf_to_parents_normalized = {}
for pdf_filename, parents in pdf_to_parents.items():
    normalized_pdf_filename = normalize_filename(pdf_filename)
    pdf_to_parents_normalized[normalized_pdf_filename] = parents

# Update variables_list with normalized filenames
for var in variables_list:
    pdf_filename = normalize_filename(var['source_pdf'])
    parents = pdf_to_parents_normalized.get(pdf_filename)
    if parents:
        var['parent_folder'] = parents['parent_folder']
        var['grandparent_folder'] = parents['grandparent_folder']
    else:
        var['parent_folder'] = 'Unknown Parent Folder'
        var['grandparent_folder'] = 'Unknown Grandparent Folder'


In [97]:
import os
import glob

pdf_base_dir = 'cbs_pdfs_with_folders'

pdf_files = glob.glob(os.path.join(pdf_base_dir, '**', '*.pdf'), recursive=True)

pdf_to_parents = {}

def normalize_filename(filename):
    filename = os.path.basename(filename)
    filename = filename.lower()
    # Remove prefixes if necessary
    prefixes = ['documentatie_', 'procedure ', 'ps ', 'ps-', 'ps_', 'lsb']
    for prefix in prefixes:
        if filename.startswith(prefix):
            filename = filename[len(prefix):]
            break
    return filename

for pdf_file in pdf_files:
    pdf_path = os.path.normpath(pdf_file)
    path_parts = pdf_path.split(os.sep)
    pdf_filename = os.path.basename(pdf_path)
    pdf_filename_normalized = normalize_filename(pdf_filename)
    if len(path_parts) >= 3:
        parent_folder = path_parts[-2]
        grandparent_folder = path_parts[-3]
    else:
        parent_folder = 'Unknown Parent Folder'
        grandparent_folder = 'Unknown Grandparent Folder'
    pdf_to_parents[pdf_filename_normalized] = {
        'parent_folder': parent_folder,
        'grandparent_folder': grandparent_folder
    }

# Now, update variables_list
for var in variables_list:
    pdf_filename = normalize_filename(var['source_pdf'])
    parents = pdf_to_parents.get(pdf_filename)
    if parents:
        var['parent_folder'] = parents['parent_folder']
        var['grandparent_folder'] = parents['grandparent_folder']
    else:
        var['parent_folder'] = 'Unknown Parent Folder'
        var['grandparent_folder'] = 'Unknown Grandparent Folder'

# Print to verify
for var in variables_list:
    print(f"Variable: {var['variable_name']}")
    print(f"Source PDF: {var['source_pdf']}")
    print(f"Parent Folder: {var['parent_folder']}")
    print(f"Grandparent Folder: {var['grandparent_folder']}\n")


Variable: BE_Id F11
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_  productiestatistiek industrie
Grandparent Folder: Industrie en energie

Variable: STATJAAR F4
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_  productiestatistiek industrie
Grandparent Folder: Industrie en energie

Variable: sbi A6
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_  productiestatistiek industrie
Grandparent Folder: Industrie en energie

Variable: kerncelcode A6
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_  productiestatistiek industrie
Grandparent Folder: Industrie en energie

Variable: RechtsvormID F3
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_  productiestatistiek industrie
Grandparent Folder: Industrie en energie

Variable: gk_SBS F6
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_  productiestatistiek industrie
Grandparent Folder: Industrie en energie

Variable: EindGewicht F8.2
Source PDF: ps-industrie.pdf
Parent Folder: Ps Industrie_

KeyError: 'variable_name'

In [98]:
variables_list

[{'variable_name': 'BE_Id F11',
  'definition': 'Bedrijfsindentificatie ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Ps Industrie_  productiestatistiek industrie',
  'grandparent_folder': 'Industrie en energie'},
 {'variable_name': 'STATJAAR F4',
  'definition': 'Verslagjaar ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Ps Industrie_  productiestatistiek industrie',
  'grandparent_folder': 'Industrie en energie'},
 {'variable_name': 'sbi A6',
  'definition': 'Standaard Bedrijfs Indeling 2008 code voor economische activiteit ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Ps Industrie_  productiestatistiek industrie',
  'grandparent_folder': 'Industrie en energie'},
 {'variable_name': 'kerncelcode A6',
  'definition': 'Code van de Kerncel ',
  'explanation': '',
  'source_pdf': 

In [99]:
# Save to a JSON file
with open("variables_list.json", "w") as file:
    json.dump(variables_list, file)

In [100]:
variables_list

[{'variable_name': 'BE_Id F11',
  'definition': 'Bedrijfsindentificatie ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Ps Industrie_  productiestatistiek industrie',
  'grandparent_folder': 'Industrie en energie'},
 {'variable_name': 'STATJAAR F4',
  'definition': 'Verslagjaar ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Ps Industrie_  productiestatistiek industrie',
  'grandparent_folder': 'Industrie en energie'},
 {'variable_name': 'sbi A6',
  'definition': 'Standaard Bedrijfs Indeling 2008 code voor economische activiteit ',
  'explanation': '',
  'source_pdf': 'ps-industrie.pdf',
  'dataset_name': 'Unknown Dataset',
  'parent_folder': 'Ps Industrie_  productiestatistiek industrie',
  'grandparent_folder': 'Industrie en energie'},
 {'variable_name': 'kerncelcode A6',
  'definition': 'Code van de Kerncel ',
  'explanation': '',
  'source_pdf': 

In [102]:
# Convert to DataFrame
import pandas as pd
df = pd.DataFrame(variables_list)

In [104]:
df 

,variable_name,definition,explanation,source_pdf,dataset_name,parent_folder,grandparent_folder
0,BE_Id F11,Bedrijfsindentificatie,,ps-industrie.pdf,Unknown Dataset,Ps Industrie_ productiestatistiek industrie,Industrie en energie
1,STATJAAR F4,Verslagjaar,,ps-industrie.pdf,Unknown Dataset,Ps Industrie_ productiestatistiek industrie,Industrie en energie
2,sbi A6,Standaard Bedrijfs Indeling 2008 code voor eco...,,ps-industrie.pdf,Unknown Dataset,Ps Industrie_ productiestatistiek industrie,Industrie en energie
3,kerncelcode A6,Code van de Kerncel,,ps-industrie.pdf,Unknown Dataset,Ps Industrie_ productiestatistiek industrie,Industrie en energie
4,RechtsvormID F3,Rechtsvormcode,,ps-industrie.pdf,Unknown Dataset,Ps Industrie_ productiestatistiek industrie,Industrie en energie
...,...,...,...,...,...,...,...
184329,"Een andere, hiervoor niet genoemde reden",Is de regeling tijdspaarfonds of tijdsparen to...,Deze variabele is een opvolger van de variabel...,spolisbus.pdf,Unknown Dataset,Spolisbus_ Banen en lonen volgens Polisadminis...,Arbeid en sociale zekerheid
184330,NaN,Aantal lopende re-integratietrajecten voor een...,Vanaf verslagperiode januari 2014 worden de WW...,uwvtrapersaggtab.pdf,Arbeid en sociale zekerheid,uwvtrapersaggtab_ Personen met een re-integrat...,Arbeid en sociale zekerheid
184331,NaN,De uitslag van het examen van een leerling.,"Een leerling kan geslaagd of afgewezen (""gezak...",examvovaktab.pdf,Unknown Dataset,Examvovaktab_ Examenuitslagen voortgezet onder...,Onderwijs
184332,Derde tijdvak,Heeft een examenkandidaat het examen van het v...,"Wanneer de code Voorlaatstjaar=L, dan heeft de...",examvovaktab.pdf,Unknown Dataset,Examvovaktab_ Examenuitslagen voortgezet onder...,Onderwijs


In [105]:
pip install googletrans==4.0.0-rc1

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 55 kB 1.7 MB/s eta 0:00:01
     |████████████████████████████████| 42 kB 3.0 MB/s eta 0:00:01
     |████████████████████████████████| 133 kB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 1.2 MB 1.9 MB/s eta 0:00:01
     |████████████████████████████████| 58 kB 3.7 MB/s eta 0:00:01
     |████████████████████████████████| 53 kB 2.5 MB/s eta 0:00:01
     |████████████████████████████████| 65 kB 4.1 MB/s eta 0:00:01
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17415 sha256=2bf90ddee7c4090198a69bc0051a702dee7028529d2f1565f96b035eefbe063d
  Stored in directory: /Users/fansantos/Library/Caches/pip/wheels/60/b3/27/d8aff3e2d5c2d0d97a117cdf0d5f13cd121e2c2b5fb49b55a0
Successfully built googletrans
  Attempting uninstall: idna
    Found existing installation: idna 3.7
    Uninstalling idna-3.7:
      Successfully uninstalled id

In [106]:
import pandas as pd
from googletrans import Translator


# Initialize the Google Translator
translator = Translator()

# Function to translate text
def translate_text(text):
    if isinstance(text, str) and text:  # Translate only non-empty strings
        return translator.translate(text, src='nl', dest='en').text
    return text

# List of columns to translate
columns_to_translate = ['definition', 'explanation', 'dataset_name', 'parent_folder', 'grandparent_folder']

# Translate each column and create new columns for the translations
for column in columns_to_translate:
    translated_column = column + "_en"  # New column name for English translation
    df[translated_column] = df[column].apply(translate_text)

print(df)

ReadTimeout: The read operation timed out